# Main Strategy Research Notebook
This notebook contains all the research underpinning the equity momentum strategy deployed on IB

It is based on data from the CRSP, daily prices for the universe of common shares exchanged on NYSE/NASDAQ/AMEX

## Data files check

In [1]:
import pandas as pd

# Just peek at the first few rows without loading everything
df_peek = pd.read_csv('/Users/hugo/quant-research/data/research/crsp_95-05.csv', nrows=5)
print(df_peek.columns.tolist())
print(df_peek.head())
print(df_peek.dtypes)

['PERMNO', 'date', 'SHRCD', 'EXCHCD', 'SICCD', 'NCUSIP', 'TICKER', 'COMNAM', 'SHRCLS', 'TSYMBOL', 'CUSIP', 'DLSTCD', 'HSICIG', 'DLRET', 'BIDLO', 'ASKHI', 'PRC', 'VOL', 'RET', 'BID', 'ASK', 'SHROUT']
   PERMNO        date  SHRCD  EXCHCD  SICCD    NCUSIP TICKER           COMNAM  \
0   10001  1995-01-03     11       3   4920  29274A10   EWST  ENERGY WEST INC   
1   10001  1995-01-04     11       3   4920  29274A10   EWST  ENERGY WEST INC   
2   10001  1995-01-05     11       3   4920  29274A10   EWST  ENERGY WEST INC   
3   10001  1995-01-06     11       3   4920  29274A10   EWST  ENERGY WEST INC   
4   10001  1995-01-09     11       3   4920  29274A10   EWST  ENERGY WEST INC   

   SHRCLS TSYMBOL  ...  HSICIG  DLRET  BIDLO  ASKHI    PRC   VOL       RET  \
0     NaN    EWST  ...     NaN    NaN  8.375   8.50  8.375   372  0.046875   
1     NaN    EWST  ...     NaN    NaN  7.500   8.25  8.000  8275 -0.044776   
2     NaN    EWST  ...     NaN    NaN  7.500   8.00  7.500  1000 -0.062500   
3 

In [5]:
df1 = pd.read_csv('/Users/hugo/quant-research/data/research/crsp_95-05.csv',
                  low_memory=False)

print(df1.shape)
print(df1.memory_usage(deep=True).sum() / 1e9, "GB")

(16312374, 22)
3.934603862 GB


In [6]:
cols_to_keep = ['PERMNO', 'date', 'SHRCD', 'EXCHCD', 'SICCD',
                'TICKER', 'COMNAM', 'PRC', 'VOL', 'RET', 'DLRET', 'SHROUT']

df1 = df1[cols_to_keep]
print(df1.memory_usage(deep=True).sum() / 1e9, "GB")

2.317180392 GB


In [7]:
df2 = pd.read_csv('/Users/hugo/quant-research/data/research/crsp_05-15.csv', low_memory=False)[cols_to_keep]
df3 = pd.read_csv('/Users/hugo/quant-research/data/research/crsp_15-25.csv', low_memory=False)[cols_to_keep]

df_crsp = pd.concat([df1, df2, df3], ignore_index=True)

# Free up memory
del df1, df2, df3

print(df_crsp.shape)
print(df_crsp.memory_usage(deep=True).sum() / 1e9, "GB")

(36840124, 12)
5.230956305 GB


In [8]:
# Essential cleaning and saving to parquet
# 1. Parse dates
df_crsp['date'] = pd.to_datetime(df_crsp['date'])

# 2. Fix negative prices (CRSP convention for bid/ask midpoint)
df_crsp['PRC'] = df_crsp['PRC'].abs()

# 3. Convert RET and DLRET to numeric (coerce 'Z', 'C' etc. to NaN)
df_crsp['RET'] = pd.to_numeric(df_crsp['RET'], errors='coerce')
df_crsp['DLRET'] = pd.to_numeric(df_crsp['DLRET'], errors='coerce')

print(df_crsp.dtypes)
print(df_crsp.shape)

PERMNO             int64
date      datetime64[us]
SHRCD              int64
EXCHCD             int64
SICCD                str
TICKER               str
COMNAM               str
PRC              float64
VOL              float64
RET              float64
DLRET            float64
SHROUT           float64
dtype: object
(36840124, 12)


In [9]:
# Check what non-numeric values are in SICCD
mask = pd.to_numeric(df_crsp['SICCD'], errors='coerce').isna() & df_crsp['SICCD'].notna()
print(df_crsp.loc[mask, 'SICCD'].unique())

<ArrowStringArray>
['Z']
Length: 1, dtype: str


In [10]:
df_crsp['SICCD'] = pd.to_numeric(df_crsp['SICCD'], errors='coerce')

In [11]:
# Keep only NYSE (1), AMEX (2), NASDAQ (3)
df_crsp = df_crsp[df_crsp['EXCHCD'].isin([1, 2, 3])]

print(df_crsp.shape)
print(f"Unique tickers: {df_crsp['TICKER'].nunique()}")
print(f"Unique PERMNOs: {df_crsp['PERMNO'].nunique()}")
print(f"Date range: {df_crsp['date'].min()} to {df_crsp['date'].max()}")

(36137567, 12)
Unique tickers: 17962
Unique PERMNOs: 16534
Date range: 1995-01-03 00:00:00 to 2024-12-31 00:00:00


In [12]:
# sanity check on NaN
print(df_crsp[['PRC', 'RET', 'DLRET', 'VOL', 'SHROUT']].isna().mean() * 100)

PRC        0.047964
RET        0.076126
DLRET     99.966857
VOL        0.047920
SHROUT     0.031792
dtype: float64


In [13]:
# Saving to parquet
crsp_path = '/Users/hugo/quant-research/data/research/crsp_clean.parquet'
df_crsp.to_parquet(crsp_path, index=False)
print("Saved successfully")

import os
size_mb = os.path.getsize(crsp_path) / 1e6
print(f"File size: {size_mb:.1f} MB")

Saved successfully
File size: 403.0 MB


In [14]:
# Quick verification before deleting anything
df_check = pd.read_parquet(crsp_path)
print(df_check.shape)
print(df_check.dtypes)

(36137567, 12)
PERMNO             int64
date      datetime64[us]
SHRCD              int64
EXCHCD             int64
SICCD            float64
TICKER               str
COMNAM               str
PRC              float64
VOL              float64
RET              float64
DLRET            float64
SHROUT           float64
dtype: object
